#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\paull\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Modelos preentrenados, visualizando con las utilidades de ultralytics

In [2]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 271.7ms
video 1/1 (frame 2/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 15.8ms
video 1/1 (frame 3/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 25.2ms
vi

In [3]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]


# Captura desde la webcam
vid = cv2.VideoCapture("C0142.MP4")
track_history = defaultdict(lambda: [])

# Para guardar el vídeo procesado para ello necesitamos el códec, los fps y el tamaño del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Guardar el vídeo procesado
out = cv2.VideoWriter("C0142_procesado.mp4", fourcc, fps, (width, height))

# Inicializamos los contadores y los sets para almacenar los IDs contados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

       # Actualiza los contadores si el ID de seguimiento no ha sido contado previamente
                if box.id is not None:
                    nombre_clase = classNames[cls]
                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

       # Dibuja los contadores en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

       # Guardar el fotograma procesado en el vídeo de salida
        out.write(img)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Libera el objeto de escritura de vídeo
out.release()
# Destruye ventanas
cv2.destroyAllWindows()



0: 384x640 1 car, 15.2ms
Speed: 20.0ms preprocess, 15.2ms inference, 1084.4ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.85
Clase --> car

0: 384x640 1 car, 27.8ms
Speed: 2.8ms preprocess, 27.8ms inference, 7.7ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 11.8ms
Speed: 1.8ms preprocess, 11.8ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 12.4ms
Speed: 1.8ms preprocess, 12.4ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.84
Clase --> car

0: 384x640 1 car, 13.6ms
Speed: 2.2ms preprocess, 13.6ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 12.5ms
Speed: 1.5ms preprocess, 12.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.81
Clase --> car

0: 384x640 1 car, 12.7ms
Speed: 2.3ms preprocess, 12.

In [13]:
from ultralytics import YOLO
import cv2

model = YOLO("yolo11n.pt")

classNames = ["person", "bicycle", "car"]

cap = cv2.VideoCapture("C0142.MP4")

# obtener las caracteristicas del video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# crear el video de salida
out = cv2.VideoWriter(
    "C0142_direccion.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# linea roja para coches y bicicletas
linea_vehiculos_y = int(height * 430 / 928)
linea_vehiculos_x1 = int(width * 500 / 1650)
linea_vehiculos_x2 = width - 5

# linea azul para personas
# esta linea atraviesa la acera en lugar de seguir su direccion
# linea azul para personas
persona_x1 = int(width * 450 / 1650)
persona_y1 = int(height * 325 / 928)

persona_x2 = int(width * 500 / 1650)
persona_y2 = int(height * 390 / 928)

# margen de seguridad
margen = 12

# contadores de direccion
flujo = {
    "person": {"acercandose": 0, "alejandose": 0},
    "bicycle": {"acercandose": 0, "alejandose": 0},
    "car": {"acercandose": 0, "alejandose": 0}
}

# guardar el ultimo lado de cada objeto
posiciones = {}

# guardar la ultima posicion de las personas
posiciones_personas = {}

# evitar contar dos veces el mismo objeto
contados = set()

while cap.isOpened():
    ret, img = cap.read()

    if not ret:
        break

    # detectar y seguir los objetos
    results = model.track(
        img,
        persist=True,
        classes=[0, 1, 2],
        verbose=False
    )

    for box in results[0].boxes:

        # comprobar que el objeto tiene identificador
        if box.id is None:
            continue

        # obtener coordenadas, identificador y clase
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        track_id = int(box.id[0])
        cls = int(box.cls[0])
        nombre = classNames[cls]

        # utilizar el centro inferior de la caja
        cx = (x1 + x2) // 2
        cy = y2

        # analizar coches y bicicletas
        if nombre in ["car", "bicycle"]:

            # comprobar si esta por encima o debajo de la linea roja
            if linea_vehiculos_x1 <= cx <= linea_vehiculos_x2:

                if cy < linea_vehiculos_y - margen:
                    lado_actual = -1
                elif cy > linea_vehiculos_y + margen:
                    lado_actual = 1
                else:
                    lado_actual = 0

                # guardar la primera posicion conocida
                if track_id not in posiciones:
                    if lado_actual != 0:
                        posiciones[track_id] = lado_actual

                # comprobar si ha cruzado la linea
                elif track_id not in contados:

                    lado_anterior = posiciones[track_id]

                    if lado_actual != 0:

                        # de arriba hacia abajo: se acerca
                        if lado_anterior == -1 and lado_actual == 1:
                            flujo[nombre]["acercandose"] += 1
                            contados.add(track_id)

                        # de abajo hacia arriba: se aleja
                        elif lado_anterior == 1 and lado_actual == -1:
                            flujo[nombre]["alejandose"] += 1
                            contados.add(track_id)

                        # actualizar el ultimo lado conocido
                        posiciones[track_id] = lado_actual

        # analizar personas en la acera
        elif nombre == "person":

            # guardar la posicion inicial del peaton
            if track_id not in posiciones_personas:
                posiciones_personas[track_id] = (cx, cy)

            elif track_id not in contados:

                # recuperar la posicion anterior
                anterior_x, anterior_y = posiciones_personas[track_id]

                # calcular en que lado de la linea esta el peaton
                lado_anterior = (
                    (persona_x2 - persona_x1) * (anterior_y - persona_y1)
                    - (persona_y2 - persona_y1) * (anterior_x - persona_x1)
                )

                lado_actual = (
                    (persona_x2 - persona_x1) * (cy - persona_y1)
                    - (persona_y2 - persona_y1) * (cx - persona_x1)
                )

                # comprobar si ha cambiado de lado
                if lado_anterior * lado_actual < 0:

                    # comprobar que el cruce se produce cerca del segmento
                    medio_x = (anterior_x + cx) // 2
                    medio_y = (anterior_y + cy) // 2

                    cerca_linea = (
                        min(persona_x1, persona_x2) - 30 <= medio_x <= max(persona_x1, persona_x2) + 30
                        and min(persona_y1, persona_y2) - 30 <= medio_y <= max(persona_y1, persona_y2) + 30
                    )

                    if cerca_linea:

                        # de un lado hacia el otro: se acerca
                        if lado_anterior > 0 and lado_actual < 0:
                            flujo["person"]["alejandose"] += 1

                        else:
                            flujo["person"]["acercandose"] += 1

                        contados.add(track_id)

                # actualizar la posicion anterior
                posiciones_personas[track_id] = (cx, cy)

        # dibujar la deteccion y el identificador
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            img,
            f"{nombre} ID:{track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    # dibujar la linea roja de vehiculos
    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y),
        (linea_vehiculos_x2, linea_vehiculos_y),
        (0, 0, 255),
        3
    )

    # dibujar los limites del margen de seguridad
    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y - margen),
        (linea_vehiculos_x2, linea_vehiculos_y - margen),
        (0, 255, 255),
        1
    )

    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y + margen),
        (linea_vehiculos_x2, linea_vehiculos_y + margen),
        (0, 255, 255),
        1
    )

    # dibujar la linea azul de personas
    cv2.line(
        img,
        (persona_x1, persona_y1),
        (persona_x2, persona_y2),
        (255, 0, 0),
        3
    )

    # mostrar los contadores
    y = 30

    for nombre in classNames:
        texto = (
            f"{nombre}: "
            f"acercandose {flujo[nombre]['acercandose']} "
            f"alejandose {flujo[nombre]['alejandose']}"
        )

        cv2.putText(
            img,
            texto,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        y += 25

    # guardar y mostrar el fotograma
    out.write(img)
    cv2.imshow("Analisis de direccion", img)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# liberar recursos
cap.release()
out.release()
cv2.destroyAllWindows()

# mostrar los resultados finales
print("Resultados del flujo:")

for nombre in classNames:
    print(nombre, flujo[nombre])

Resultados del flujo:
person {'acercandose': 1, 'alejandose': 6}
bicycle {'acercandose': 0, 'alejandose': 0}
car {'acercandose': 14, 'alejandose': 6}


Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [ ]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()